# SomaliTTS — تشغيل محرك Hubi مؤقتًا على Kaggle GPU

هذه جلسة تجريبية مؤقتة. اختر GPU من إعدادات Notebook وشغّل الإنترنت. لا تجعل الدفتر عامًا ولا تشارك رابط النفق أو الرمز. عند إيقاف الجلسة ينقطع المحرك.

In [ ]:
!pip -q install fastapi 'uvicorn[standard]' python-multipart transformers huggingface_hub soundfile
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
!chmod +x /kaggle/working/cloudflared
!git clone --depth 1 --branch feature/hubi-inference-api https://github.com/abdulahim294-ai/SomaliTTS.git /kaggle/working/SomaliTTS


## تشغيل النموذج وإنشاء رابط HTTPS مؤقت

شغّل الخلية التالية مرة واحدة. انتظر حتى يظهر `HUBI READY`. انسخ رابط الخدمة والرمز السري إلى إعدادات الخادم في مشروع Somali Voice Studio فقط. لا ترسل الرمز في المحادثة.

In [ ]:
import os, sys, time, secrets, subprocess, urllib.request, re
os.environ['HUBI_API_TOKEN'] = secrets.token_urlsafe(48)
os.environ['HUBI_MODEL_ID'] = 'ayeeyokoris/hubi-tts'
os.environ['HUBI_MODEL_REVISION'] = 'main'  # experimental; pin a reviewed commit before production
sys.path.insert(0, '/kaggle/working/SomaliTTS/hubi-inference')
os.chdir('/kaggle/working/SomaliTTS/hubi-inference')
api = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'app:app', '--host', '127.0.0.1', '--port', '7860'], stdout=open('/kaggle/working/hubi-api.log','w'), stderr=subprocess.STDOUT, env=os.environ.copy())
print('Loading Hubi model on GPU. This may take several minutes...')
ready = False
for _ in range(360):
    if api.poll() is not None: break
    try:
        with urllib.request.urlopen('http://127.0.0.1:7860/health', timeout=3) as r:
            if r.status == 200:
                import json
                data = json.loads(r.read().decode())
                if data.get('ready'):
                    ready = True; break
    except Exception: pass
    time.sleep(5)
if not ready:
    print(open('/kaggle/working/hubi-api.log').read()[-12000:])
    raise RuntimeError('Hubi did not become ready. Check GPU and internet settings.')
tunnel = subprocess.Popen(['/kaggle/working/cloudflared','tunnel','--no-autoupdate','--url','http://127.0.0.1:7860'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for _ in range(90):
    line = tunnel.stdout.readline()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group(0); break
if not public_url:
    raise RuntimeError('Could not create Cloudflare tunnel. Check notebook internet access.')
print('\nHUBI READY')
print('HUBI_TTS_URL =', public_url)
print('HUBI_TTS_TOKEN =', os.environ['HUBI_API_TOKEN'])
print('\nKeep this notebook session running. Treat both values as passwords; do not share screenshots containing them.')

## إيقاف الخدمة

أوقف جلسة Kaggle عندما تنتهي. الرابط مؤقت ويتغير عند إعادة التشغيل. لا تحفظ مخرجات الخلية في دفتر عام لأن فيها رمز الوصول.